# 01 — Ingest Bronze

## Objective

Read raw source files from the GitHub repository, add technical ingestion metadata, and store them as Bronze Delta tables in Databricks.

## This notebook performs

- Installation of required libraries for file reading.
- Reading raw CSV, JSON and Excel files from GitHub.
- Conversion from pandas DataFrames to Spark DataFrames.
- Addition of Bronze metadata columns:
  - `source_file`
  - `ingestion_timestamp`
  - `bronze_load_id`
- Creation of the project schema.
- Storage of Bronze tables as Delta tables.
- Basic row count verification.

## Notes

The Bronze layer preserves the raw business data without applying business cleaning or standardization. Data quality issues are intentionally kept at this stage and will be handled in the Silver transformation layer.

In [0]:


%pip install openpyxl

import uuid
import pandas as pd

from pyspark.sql import functions as F

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Base URL for raw files stored in the GitHub repository

RAW_BASE_URL = "https://raw.githubusercontent.com/agustinaromeroartaza/workbridge-workforce-analytics/master/data/raw"

# Unique ID for this ingestion execution

bronze_load_id = str(uuid.uuid4())

print(f"Bronze load ID: {bronze_load_id}")

Bronze load ID: e6881281-d99b-4b4a-b7ed-846546bcd8f3


In [0]:
# Read raw files from GitHub using pandas

headcount_pdf = pd.read_csv(f"{RAW_BASE_URL}/headcount.csv")
hours_pdf = pd.read_csv(f"{RAW_BASE_URL}/hours.csv")
goals_pdf = pd.read_csv(f"{RAW_BASE_URL}/goals.csv")
costs_pdf = pd.read_json(f"{RAW_BASE_URL}/costs.json")

assignments_pdf = pd.read_excel(f"{RAW_BASE_URL}/assignments.xlsx")
clients_pdf = pd.read_excel(f"{RAW_BASE_URL}/clients.xlsx")

In [0]:
# Convert pandas DataFrames to Spark DataFrames

headcount_df = spark.createDataFrame(headcount_pdf)
hours_df = spark.createDataFrame(hours_pdf)
goals_df = spark.createDataFrame(goals_pdf)
costs_df = spark.createDataFrame(costs_pdf)
assignments_df = spark.createDataFrame(assignments_pdf)
clients_df = spark.createDataFrame(clients_pdf)

In [0]:

# Add technical metadata columns to raw ingested data.

# These columns help track where each record came from and when it was loaded.

def add_bronze_metadata(df, source_file):
  
    return (
        df
        .withColumn("source_file", F.lit(source_file))
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("bronze_load_id", F.lit(bronze_load_id))
    )

In [0]:
# Add Bronze metadata to each raw dataset

bronze_headcount_df = add_bronze_metadata(headcount_df, "headcount.csv")
bronze_assignments_df = add_bronze_metadata(assignments_df, "assignments.xlsx")
bronze_hours_df = add_bronze_metadata(hours_df, "hours.csv")
bronze_costs_df = add_bronze_metadata(costs_df, "costs.json")
bronze_goals_df = add_bronze_metadata(goals_df, "goals.csv")
bronze_clients_df = add_bronze_metadata(clients_df, "clients.xlsx")

In [0]:
# Create and use project schema

spark.sql("CREATE SCHEMA IF NOT EXISTS workbridge")
spark.sql("USE SCHEMA workbridge")

DataFrame[]

In [0]:
# Save Bronze tables as managed Delta tables

bronze_headcount_df.write.mode("overwrite").format("delta").saveAsTable("bronze_headcount")
bronze_assignments_df.write.mode("overwrite").format("delta").saveAsTable("bronze_assignments")
bronze_hours_df.write.mode("overwrite").format("delta").saveAsTable("bronze_hours")
bronze_costs_df.write.mode("overwrite").format("delta").saveAsTable("bronze_costs")
bronze_goals_df.write.mode("overwrite").format("delta").saveAsTable("bronze_goals")
bronze_clients_df.write.mode("overwrite").format("delta").saveAsTable("bronze_clients")

In [0]:
# Check row counts for Bronze tables

tables = [
    "bronze_headcount",
    "bronze_assignments",
    "bronze_hours",
    "bronze_costs",
    "bronze_goals",
    "bronze_clients",
]

for table in tables:
    count = spark.table(table).count()
    print(f"{table}: {count} rows")

bronze_headcount: 508 rows
bronze_assignments: 555 rows
bronze_hours: 4154 rows
bronze_costs: 4932 rows
bronze_goals: 576 rows
bronze_clients: 15 rows
